In [ ]:
import os 

os.getcwd()

In [ ]:
NAMING_MAPPING =  {"baselines_5.2.1": "SketchBoost", "hyperbolic_5.2.1": "HASBoost (lin)", "sigmoid_5.2.1": "HASBoost (sig)", 
                   "baselines_5.2.1": "SketchBoost", 
                    
                "get_weights_calls": 'Вызовы get_weights',
               "xgboost_5.2.1": "XGBoost", "lgbm_5.2.1": "LightGBM", "ntrees": "Число деревьев",
               'mean_leaves': 'Среднее число листьев',
                   'sketch_proportion': 'Общая доля от времени обучения, %',
    'get_weights_avg_time': 'Среднее время одной операции, мс',
    'experiment': 'Эксперимент',
    'dataset': "Датасет",
    'lr': "Скорость обучения",
    "subsample": "Доля объектов", 
    "sketch_outputs": "Количество выходов"
               }

In [ ]:
def snake_to_capitalized_spaced_v2(snake_str):
    """
    Alternative implementation using replace and title().
    """
    # Replace underscores with spaces and then apply title case
    return snake_str.replace('_', ' ').title()

In [ ]:
COMPUTATIONAL_METRICS = ['duration_seconds', 'inference_time', 'mean_leaves', 'mean_nodes', 'ntrees',
        'train_time', 'get_weights_avg_time', 'get_indexers_total_time', 'get_weights_calls', 'get_weights_total_time, get_indexers_avg_time', 'get_indexers_calls']
DETAILED_COMPUTATIONAL = ['get_weights_avg_time', 'get_indexers_total_time', 'get_weights_calls', 'get_weights_total_time, get_indexers_avg_time', 'get_indexers_calls']

In [ ]:
def filter_df(df, rule):
    for k, v in rule.items():
        if k not in df.columns:
            continue
        df = df[df[k] == v]
    return df

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def get_all_runs_data(experiment_names=None, include_artifacts=False):
    """
    Extract all runs data from MLflow experiments into a comprehensive DataFrame
    
    Args:
        experiment_names: List of experiment names or None for all experiments
        include_artifacts: Whether to include artifact URIs
    
    Returns:
        DataFrame with all runs data
    """
    client = MlflowClient()
    
    # Get experiments
    if experiment_names is None:
        experiments = client.search_experiments()
    else:
        experiments = [client.get_experiment_by_name(name) for name in experiment_names]
        experiments = [exp for exp in experiments if exp is not None]
    
    all_runs_data = []
    
    for experiment in tqdm(experiments, desc="Processing experiments"):
        experiment_id = experiment.experiment_id
        experiment_name = experiment.name
        
        print(f"Processing experiment: {experiment_name}")
        
        # Get all runs for this experiment
        runs = client.search_runs(
            experiment_ids=[experiment_id],
            max_results=10000  # Adjust if you have more runs
        )
        
        for run in tqdm(runs, desc=f"Runs in {experiment_name}", leave=False):
            run_data = {
                'run_id': run.info.run_id,
                'experiment_id': experiment_id,
                'experiment_name': experiment_name,
                'run_name': run.data.tags.get('mlflow.runName', ''),
                'status': run.info.status,
                'start_time': pd.to_datetime(run.info.start_time, unit='ms'),
                'end_time': pd.to_datetime(run.info.end_time, unit='ms') if run.info.end_time else None,
                'duration_seconds': (run.info.end_time - run.info.start_time) / 1000.0 if run.info.end_time and run.info.start_time else None,
            }
            
            # Add parameters
            for key, value in run.data.params.items():
                run_data[f'param_{key}'] = value
            
            # Add metrics
            for key, value in run.data.metrics.items():
                run_data[f'metric_{key}'] = value
            
            # Add tags
            for key, value in run.data.tags.items():
                if key not in ['mlflow.runName', 'mlflow.user']:
                    run_data[f'tag_{key}'] = value
            
            # Add artifact location if requested
            if include_artifacts:
                run_data['artifact_uri'] = run.info.artifact_uri
            
            all_runs_data.append(run_data)

    runs_df = pd.DataFrame(all_runs_data)
    if runs_df.empty:
        return runs_df

    named_runs = runs_df[runs_df['run_name'] != '']
    unnamed_runs = runs_df[runs_df['run_name'] == '']
    latest_named_runs = (
        named_runs.sort_values('start_time')
        .drop_duplicates(subset=['run_name'], keep='last')
    )

    return pd.concat([latest_named_runs, unnamed_runs], ignore_index=True)

def get_baselines(path, version):
    start_dir = os.getcwd()
    os.chdir(path)
    df = get_all_runs_data(['baselines_' + str(version)])
    os.chdir(start_dir)
    return df 


In [ ]:
# bsln = get_baselines('/home/leostre/Рабочий стол/py-boost/5.2', '5.2')

In [ ]:
EXCLUDE = {'experiment_id', 'experiment_name', 'status', 'start_time', 'end_time', 'run_name', 'tag_mlflow.source.name', 'tag_mlflow.source.git.commit',
       'tag_mlflow.source.type', 'param_error', 'tag_status',
       'param_total_runs', 'param_successful_runs', 'mean_f1', 'mean_accuracy', 'param_n_splits', 'param_n_successful_folds'}

def filter_data(data):
    after_exclusion_by_name = [
        col for col in data.columns if col not in EXCLUDE
    ]
    print(after_exclusion_by_name)
    statistics = ('mean', 'max', 'min', 'std', 'median')
    after_exclusion_agg = [
        col for col in after_exclusion_by_name if 'leaves' in col or 'nodes' in col or
        'tree' in col or
        not any(statistic in col for statistic in statistics) and not 'metric_fold' in col or col in ('param_stabilization_threshold', 'param_smoothing_alpha')
    ]
    filtered_data = data[after_exclusion_agg 
                        #  + ['metric_std_num_trees', 'metric_mean_num_trees',]
                         ]
    filtered_pivot = filtered_data.rename(columns={col: col.removeprefix('param_') for col in filtered_data.columns})
    return filtered_pivot

def melt_metrics(df):
    # Identify metric columns
    metric_cols = [col for col in df.columns if col.startswith('metric_')]
    print(metric_cols)
    
    # Identify ID columns (all non-metric columns)
    id_cols = [col for col in df.columns if not col.startswith('metric_')]
    print(id_cols)
    
    # Melt using pandas melt (more control)
    melted_df = df.melt(
        id_vars=id_cols,
        value_vars=metric_cols,
        var_name='metric_fold',
        value_name='value'
    )
    
    # Extract metric name and fold number using regex pattern
    pattern = r'metric_(.+?)_fold_(\d+)$'
    extracted = melted_df['metric_fold'].str.extract(pattern)
    
    # Create new columns
    melted_df['metric'] = extracted[0]
    melted_df['fold'] = extracted[1]
    
    # For metrics without fold numbers (like 'metric_total_training_time')
    # Fill NaN metric names with the original string without 'metric_' prefix
    mask = melted_df['metric'].isna()
    melted_df.loc[mask, 'metric'] = melted_df.loc[mask, 'metric_fold'].str.replace('metric_', '')
    
    # Drop the temporary column and clean up
    melted_df = melted_df.drop('metric_fold', axis=1)
    melted_df = melted_df.reset_index(drop=True)
    
    return melted_df


In [ ]:
def flatten_index(df):
    columns = df.columns
    if isinstance(columns, pd.MultiIndex):
        columns = [c[0] if not c[1] else c[1] for c in columns] 
        columns = [c if c != 'mean' else 'value' for c in columns]
    df.columns = columns  

## Compare experiments

In [ ]:

TO_INT = ['sketch_outputs', 'get_weights_calls', 'get_indexers_calls']
TO_FLOAT = ['smoothing_alpha', 'stabilization_threshold', 'subsample', 'lr', ]

def process_mlruns(processed, EXPS, DATASET, path='.'):
    curdir = os.getcwd()
    os.chdir(path)
    dfs = [get_all_runs_data([exp]) for exp in EXPS] 

    version = None # '5.2'
    if version:
        dfs += [get_baselines(f'../{version}', version)]
        EXPS += [f'baselines_{version}']

    for exp, df in zip(EXPS, dfs):
        if DATASET:
            df = df[df.param_dataset.isin(DATASET)]
        df = df.rename(columns={'param_learning_rate': 'param_lr'})
        print(exp, df.shape)
        df = filter_data(df)
        mlt_df = melt_metrics(df)
        print(exp, mlt_df.shape)
        agg_mtrs = mlt_df.groupby(['dataset',
                                    *(['sketch_method', 'sketch_outputs'] if 'sketch_method' in mlt_df.columns else []), 
                                    *(['smoothing_alpha', 'stabilization_threshold'] if 'smoothing_alpha' in mlt_df.columns else []),
                                    'subsample', 'lr', 'metric', ]).agg({'value': ['mean', 'std']})#.reset_index()
        flatten_index(agg_mtrs)
        for c in TO_FLOAT:
            if not c in agg_mtrs:
                continue
            agg_mtrs[c] = agg_mtrs[c].astype(float)
        for c in TO_INT:
            if not c in agg_mtrs:
                continue
            agg_mtrs[c] = agg_mtrs[c].astype(int)
        agg_mtrs['experiment'] = exp
        processed[exp] = (agg_mtrs).reset_index()
    os.chdir(curdir)
    return processed

Specify the folder with `mlruns`

In [ ]:
DATASET = None

In [ ]:
processed = process_mlruns({}, [
    'sigmoid_5.2.1',
    'hyperbolic_5.2.1',
    'baselines_5.2.1',
    'lgbm_5.2.1',
    'xgboost_5.2.1',
], DATASET, '/home/leostre/Рабочий стол/py-boost/hasboost_upd_22.04')

In [ ]:
processed2 = process_mlruns({}, [
    'sigmoid_5.2.1',
    'hyperbolic_5.2.1',
    'baselines_5.2.1',
    # 'lgbm_5.2.1',
    # 'xgboost_5.2.1',
    # 'sigmoid_5.0',
    # 'hyperbolic_4.0',
], DATASET, '../dump_27.05')

In [ ]:
all_data = pd.concat([df.reset_index() for df in
    processed.values()
], axis=0)

all_data = pd.concat([all_data] + [df.reset_index() for df in
    processed2.values()
], axis=0)


In [ ]:
hamming_loss = all_data[(all_data.metric == 'accuracy')].copy() 
hamming_loss['value'] = 1 - hamming_loss['value']
hamming_loss['metric'] = 'hamming_loss'

all_data = pd.concat(
    [all_data, hamming_loss], axis=0
)

In [ ]:
IN_PERCENT = True

if IN_PERCENT:
    all_data.loc[(~all_data.metric.isin(set(COMPUTATIONAL_METRICS) | set(DETAILED_COMPUTATIONAL))) & (~all_data.metric.str.contains('oss')), 'value'] *= 100

In [ ]:
all_data

## Main comparison

In [ ]:
METRICS_FOR_TABLE = ['roc_auc', 'f1', 'accuracy', 
                     'train_time', 'inference_time'
                    # 'bce_loss', 'multiclass_logloss', 
                     ]
BASELINE_EXPERIMENTS = ['baselines_5.2.1']

DATASET_FOR_TABLE = {'genbase': {'stabilization_threshold': '0.5', 'lr': '0.1'},
                      'mediamill': {'stabilization_threshold': '1.0', 'lr': '0.1'},
                        'yeast': {'stabilization_threshold': '0.5', 'lr': '0.005'}, 
                        'rt_iot2022': {'stabilization_threshold': '1.0', 'lr': '0.1'}, 
                        'birds': {'stabilization_threshold': '1.0', 'lr': '0.1'}}

In [ ]:
def agg_mean_std(x):
    return f'{np.mean(x):.2f}±{np.std(x):.2f}'

from functools import partial 

In [ ]:
def create_sketch_boost_comparison(metrics=None, baselines=None, datasets=None):
    metrics = metrics or METRICS_FOR_TABLE
    baselines = baselines or BASELINE_EXPERIMENTS
    tables_for_datasets = []
    datasets = datasets or DATASET_FOR_TABLE
    for dataset, hps in datasets.items():
        params = {'dataset': dataset} | hps 
        proper_data = filter_df(all_data, params)
        proper_data = proper_data[proper_data.metric.isin(metrics)]
        # adding baseline
        baseline_params = {**params}
        baseline_params.pop('stabilization_threshold')
        baseline_data = filter_df(all_data, baseline_params)
        baseline_data = baseline_data[(baseline_data.experiment.isin(baselines) & baseline_data.metric.isin(metrics))]

        pv_mtrs = pd.pivot_table(pd.concat([proper_data, baseline_data], axis=0), index=['dataset', 'experiment'], columns='metric', values='value', aggfunc=agg_mean_std).reset_index()
        tables_for_datasets.append(pv_mtrs)
    concat_tables = pd.concat(tables_for_datasets, axis=0)
    
    concat_tables.rename(columns=NAMING_MAPPING, inplace=True)
    ORDERING = {
    'HASBoost (sig)': 0, 
    'HASBoost (lin)': 1,
    'SketchBoost': 2, 
    'LightGBM': 3, 
    'XGBoost': 4
    }
    def _wrap(x):
        return x[0], ORDERING[x[1]]
    concat_tables = concat_tables.map(lambda x: NAMING_MAPPING.get(x, x))

    concat_tables.set_index(['Датасет', 'Эксперимент'], inplace=True)
    # concat_tables.sort_index(level=[0, 1], key=lambda idx: idx.map(_wrap), inplace=True)
    concat_tables = concat_tables[metrics]

    return concat_tables
    
    


In [ ]:
create_sketch_boost_comparison(baselines=['baselines_5.2.1', 'xgboost_5.2.1', 'lgbm_5.2.1'])#.to_excel('Сравнение со SketchBoost.xlsx')#.to_csv('Сравнение со SketchBoost.csv')

In [ ]:
DATASET_FOR_TABLE2 = {
                        # 'delicious': {'stabilization_threshold': '1.0', 'lr': '0.1'}, 
                        'cifar10': {'stabilization_threshold': '0.5', 'lr': '0.005'}, 
                        'age_prediction': {'stabilization_threshold': '0.5', 'lr': '0.1'},
                        'mnist': {'stabilization_threshold': '0.5', 'lr': '0.005'}, 
                        'moa': {'stabilization_threshold': '1.0', 'lr': '0.005'},
                        'delicious': {'stabilization_threshold': '1.0', 'lr': '0.005'},
                        }
# METRICS_FOR_TABLE = ['roc_auc', 'f1', 'accuracy', 'train_time', 'inference_time']
BASELINE_EXPERIMENTS2 = ['baselines_5.2.1', 'xgboost_5.2.1', 'lgbm_5.2.1']

In [ ]:
create_sketch_boost_comparison(datasets=DATASET_FOR_TABLE2, baselines=BASELINE_EXPERIMENTS2).to_excel('Открытые бенчмарки 1.xlsx')#.to_csv('Открытые бенчмарки 1.csv')

In [ ]:
create_sketch_boost_comparison(datasets=DATASET_FOR_TABLE2, metrics= ['roc_auc', 'f1', 'accuracy', 
                    'bce_loss', 'multiclass_logloss', 
                     ], baselines=BASELINE_EXPERIMENTS).to_excel('по запросу Андрея.xlsx')#.to_csv('Открытые бенчмарки 1.csv')

In [ ]:
os.getcwd()